# Data Cleaning

This notebook will be used for data profiling and cleaning.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
from src.paths import RAW_DIR, INTERIM_DIR, PROCESSED_DIR
from src.data_loader import load_data
from src.config import RANDOM_SEED

In [ ]:
df = load_data(
    "merged_order_items",
    stage="interim"
)

In [ ]:
df.tail()

# 1. Data Type Validation

In [ ]:
df.info()

In [ ]:
duplicate_items = df.duplicated(
    subset=["order_id", "order_item_id"]
).sum()

print("Duplicate order-items:", duplicate_items)

assert duplicate_items == 0

In [ ]:
df["order_status"].value_counts(dropna=False)

Currently all the columns are of the correct data type, but we will need to convert to categorical data type for analysis and modeling after the data cleaning process.
- order_status
- customer_state
- seller_state
- product_category_name


In [ ]:
for col in [
    "order_status",
    "customer_city",
    "seller_city",
    "product_category_name",
]:
    original_unique = df[col].nunique(dropna=True)
    normalized_unique = (
        df[col]
        .dropna()
        .str.strip()
        .str.lower()
        .nunique()
    )

    print(
        f"{col}: "
        f"{original_unique} -> {normalized_unique}"
    )

In [ ]:
lower_cols = [
    "order_status",
    "customer_city",
    "seller_city",
    "product_category_name",
]

upper_cols = [
    "customer_state",
    "seller_state",
]

for col in lower_cols + upper_cols:
    s = df[col]

    whitespace = (
        s.notna()
        & (s != s.str.strip())
    ).sum()

    blank = (
        s.notna()
        & s.str.strip().eq("")
    ).sum()

    special_space = (
        s.fillna("")
        .str.contains(r"[\t\n\r]", regex=True)
        .sum()
    )

    print(
        f"{col}: whitespace={whitespace}, "
        f"blank={blank}, tab/newline={special_space}"
    )

In [ ]:
df[lower_cols + upper_cols] = (
    df[lower_cols + upper_cols]
    .replace(r"^\s*$", pd.NA, regex=True)
)

In [ ]:
for col in lower_cols:
    df[col] = (
        df[col]
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.lower()
    )

for col in upper_cols:
    df[col] = (
        df[col]
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.upper()
    )

In [ ]:
for col in lower_cols:
    leading_trailing = (
        df[col].notna()
        & (df[col] != df[col].str.strip())
    ).sum()

    uppercase_remaining = (
        df[col]
        .dropna()
        .str.contains(r"[A-Z]", regex=True)
        .sum()
    )

    print(
        f"{col}: "
        f"whitespace={leading_trailing}, "
        f"uppercase={uppercase_remaining}"
    )

In [ ]:
for col in upper_cols:
    invalid_case = (
        df[col]
        .dropna()
        .ne(df[col].dropna().str.upper())
        .sum()
    )

    print(f"{col}: {invalid_case}")

After standardization, no leading/trailing whitespace or case inconsistencies remained.

# 2. Business Validation

In [ ]:
orders_check = df.drop_duplicates("order_id").copy()

orders_check["approval_before_purchase"] = (
    orders_check["order_approved_at"]
    < orders_check["order_purchase_timestamp"]
)

orders_check["carrier_before_approval"] = (
    orders_check["order_delivered_carrier_date"]
    < orders_check["order_approved_at"]
)

orders_check["delivery_before_carrier"] = (
    orders_check["order_delivered_customer_date"]
    < orders_check["order_delivered_carrier_date"]
)

orders_check["delivery_before_purchase"] = (
    orders_check["order_delivered_customer_date"]
    < orders_check["order_purchase_timestamp"]
)

timeline_flags = [
    "approval_before_purchase",
    "carrier_before_approval",
    "delivery_before_carrier",
    "delivery_before_purchase",
]

print(orders_check[timeline_flags].sum())

In [ ]:
hard_invalid = (
    orders_check["approval_before_purchase"]
    | orders_check["delivery_before_carrier"]
    | orders_check["delivery_before_purchase"]
)

invalid_order_ids = orders_check.loc[
    hard_invalid,
    "order_id"
]

print("Hard-invalid orders removed:", len(invalid_order_ids))

df = df[
    ~df["order_id"].isin(invalid_order_ids)
].copy()

In [ ]:
product_numeric_cols = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
]

for col in product_numeric_cols:
    df.loc[df[col] <= 0, col] = pd.NA

# 3. Handling Missing Values

In [ ]:
df.isna().sum()

In [ ]:
df["missing_category"] = (
    df["product_category_name"].isna()
)

df["missing_product_features"] = (
    df[
        [
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm",
        ]
    ]
    .isna()
    .any(axis=1)
)

In [ ]:
print("Missing category:", df["missing_category"].sum())
print(
    "Missing product features:",
    df["missing_product_features"].sum()
)

# 4. Detect Outliers & Suspicious Values

In [ ]:
def check_iqr(series):
    s = series.dropna()

    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outliers = s[(s < lower) | (s > upper)]

    return lower, upper, len(outliers)

In [ ]:
order_level = (
    df.groupby("order_id")
    .agg(
        order_status=("order_status", "first"),
        customer_id=("customer_id", "first"),

        order_purchase_timestamp=(
            "order_purchase_timestamp", "first"
        ),
        order_approved_at=(
            "order_approved_at", "first"
        ),
        order_delivered_carrier_date=(
            "order_delivered_carrier_date", "first"
        ),
        order_delivered_customer_date=(
            "order_delivered_customer_date", "first"
        ),
        order_estimated_delivery_date=(
            "order_estimated_delivery_date", "first"
        ),

        customer_city=("customer_city", "first"),
        customer_state=("customer_state", "first"),

        order_value=("price", "sum"),
        freight_value=("freight_value", "sum"),
        item_count=("order_item_id", "count"),
    )
    .reset_index()
)

In [ ]:
for col in ["order_value", "freight_value"]:
    lower, upper, n = check_iqr(order_level[col])

    print(
        f"{col}: lower={lower:.2f}, "
        f"upper={upper:.2f}, outliers={n}"
    )

order_level["item_count"].describe()

In [ ]:
carrier_before_approval_ids = orders_check.loc[
    orders_check["carrier_before_approval"],
    "order_id"
]

order_level["carrier_before_approval"] = (
    order_level["order_id"]
    .isin(carrier_before_approval_ids)
)

In [ ]:
late_delivery_df = order_level[
    (order_level["order_status"] == "delivered")
    & order_level["order_delivered_customer_date"].notna()
    & order_level["order_estimated_delivery_date"].notna()
].copy()

print(
    "Orders used for late-delivery analysis:",
    len(late_delivery_df)
)

In [ ]:
late_delivery_df["delivery_days"] = (
    late_delivery_df["order_delivered_customer_date"]
    - late_delivery_df["order_purchase_timestamp"]
).dt.total_seconds() / 86400

late_delivery_df["delay_days"] = (
    late_delivery_df["order_delivered_customer_date"]
    - late_delivery_df["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

late_delivery_df["is_late"] = (
    late_delivery_df["delay_days"] > 0
)

In [ ]:
for col in ["delivery_days", "delay_days"]:
    lower, upper, n = check_iqr(late_delivery_df[col])

    print(
        f"{col}: lower={lower:.2f}, "
        f"upper={upper:.2f}, outliers={n}"
    )

Some really late deliveries are likely to be outliers, but we will keep them for analysis. We will also keep the negative delays, as they may be due to early deliveries.

In [ ]:
for col in ["price", "freight_value"]:
    lower, upper, n = check_iqr(df[col])

    print(
        f"{col}: lower={lower:.2f}, "
        f"upper={upper:.2f}, outliers={n}"
    )

In [ ]:
products = df.drop_duplicates("product_id")

product_cols = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
]

for col in product_cols:
    lower, upper, n = check_iqr(products[col])

    print(
        f"{col}: lower={lower:.2f}, "
        f"upper={upper:.2f}, outliers={n}"
    )

In [ ]:
assert (df["price"] > 0).all(), "Invalid non-positive price found"
assert (df["freight_value"] >= 0).all(), "Invalid negative freight found"

print("Price and freight values are valid.")

# 5. Final Validation

In [ ]:
print("Final item-level shape:", df.shape)

print(
    "Duplicate order-items:",
    df.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum()
)

print(
    "Invalid price:",
    (df["price"] <= 0).sum()
)

print(
    "Invalid freight:",
    (df["freight_value"] < 0).sum()
)

In [ ]:
print("Order-level shape:", order_level.shape)

print(
    "Duplicate orders:",
    order_level["order_id"].duplicated().sum()
)

print(
    "Unique orders:",
    order_level["order_id"].nunique()
)

In [ ]:
print(
    "Late-delivery orders:",
    len(late_delivery_df)
)

print(
    "Missing actual delivery date:",
    late_delivery_df[
        "order_delivered_customer_date"
    ].isna().sum()
)

print(
    "Missing estimated delivery date:",
    late_delivery_df[
        "order_estimated_delivery_date"
    ].isna().sum()
)

print(
    "Duplicate orders:",
    late_delivery_df["order_id"]
    .duplicated()
    .sum()
)

In [ ]:
expected_is_late = (
    late_delivery_df["delay_days"] > 0
)

print(
    "Incorrect is_late:",
    (
        late_delivery_df["is_late"]
        != expected_is_late
    ).sum()
)

# 6. Save Processed Data

In [ ]:
clean_items_path = (
    PROCESSED_DIR / "cleaned_order_items.parquet"
)

df.to_parquet(
    clean_items_path,
    index=False
)

print("Saved:", clean_items_path)

In [ ]:
late_delivery_path = (
    PROCESSED_DIR / "late_delivery_orders.parquet"
)

late_delivery_df.to_parquet(
    late_delivery_path,
    index=False
)

print("Saved:", late_delivery_path)

In [ ]:
print(clean_items_path.exists())
print(late_delivery_path.exists())